In [52]:
from pyspark.sql import SparkSession

In [53]:

spark =    SparkSession.builder \
    .appName("day9") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [54]:
spark.sparkContext.setLogLevel("WARN")

transformation

In [55]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, TimestampNTZType, LongType
)

In [56]:
# ── 1. EXPLICIT SCHEMA ────────────────────────────────────────────────────────
# Never use inferSchema in production.
# inferSchema scans the file TWICE — one pass to guess types, one to read data.
# At 100GB that extra scan costs real time and money.
# Explicit schema also enforces types — a salary="N/A" becomes null, not a crash.
schema = StructType([
    StructField("VendorID",    IntegerType(), nullable=False),
    StructField("tpep_pickup_datetime",      TimestampNTZType(),  nullable=True),
    StructField("tpep_dropoff_datetime",     TimestampNTZType(),  nullable=True),
    StructField("passenger_count",     LongType(),  nullable=True),
    StructField("trip_distance",       DoubleType(), nullable=True),
    StructField("RatecodeID",   LongType(),  nullable=True),
    StructField("store_and_fwd_flag",    StringType(),  nullable=True),
    StructField("PULocationID",      IntegerType(),  nullable=True),
    StructField("DOLocationID", IntegerType(),    nullable=True),
    StructField("payment_type",       LongType(), nullable=True),
    StructField("fare_amount",   DoubleType(),  nullable=True),
    StructField("extra",    DoubleType(),  nullable=True),
    StructField("mta_tax",      DoubleType(),  nullable=True),
    StructField("tip_amount", DoubleType(),   nullable=True),
    StructField("improvement_surcharge",    DoubleType(),  nullable=True),
    StructField("total_amount",      DoubleType(),  nullable=True),
    StructField("congestion_surcharge", DoubleType(),    nullable=True),
    StructField("Airport_fee",    DoubleType(),  nullable=True),
    StructField("cbd_congestion_fee",      DoubleType(),  nullable=True),
])

In [57]:
df = spark.read.schema(schema) \
    .parquet("s3a://raw-data/yellow_tripdata_2025*")

In [58]:
print(f"\n[RAW] Rows: {df.count()}")


[RAW] Rows: 48722602


In [59]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [60]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [62]:
# take much time to process for all data
#df.summary().show()
#df.summary('count').show() # choose some agg function [count, mean, stddev, min, 25%, 50%, 75%, max] to process data

# might specify some column
df.select("fare_amount").describe().show()

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|          48722602|
|   mean| 18.41937561646988|
| stddev|142.84606775530187|
|    min|           -1807.6|
|    max|         863372.12|
+-------+------------------+



In [63]:
# where and filter work the same way
df.where( df.store_and_fwd_flag.isNull() ).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-05-01 00:05:05|  2025-05-01 00:34:27|           NULL|         8.12|      NULL|              NULL|          90|          42|           0|      34.14|  0.0

In [64]:
# count rows in June 2025
df.where(df.tpep_pickup_datetime.startswith('2025-06')).count()

4322949

In [65]:
# don't make column name short with ...
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-05-01 00:07:06 |2025-05-01 00:24:15  |1              |3.7          |1         |N                 |140         |202         |1           |18.4       |4.25 

In [66]:
df.select(
    F.min(df.tpep_dropoff_datetime),
    F.max(df.tpep_dropoff_datetime)
).show()

+--------------------------+--------------------------+
|min(tpep_dropoff_datetime)|max(tpep_dropoff_datetime)|
+--------------------------+--------------------------+
|       2007-12-05 19:02:00|       2026-01-05 13:00:34|
+--------------------------+--------------------------+



In [67]:
df.select(
    F.min(df.tpep_dropoff_datetime),
    F.dayofmonth(F.min(df.tpep_dropoff_datetime)),
    F.month(F.min(df.tpep_dropoff_datetime)),
    F.year(F.min(df.tpep_dropoff_datetime)),
).show()

+--------------------------+--------------------------------------+---------------------------------+--------------------------------+
|min(tpep_dropoff_datetime)|dayofmonth(min(tpep_dropoff_datetime))|month(min(tpep_dropoff_datetime))|year(min(tpep_dropoff_datetime))|
+--------------------------+--------------------------------------+---------------------------------+--------------------------------+
|       2007-12-05 19:02:00|                                     5|                               12|                            2007|
+--------------------------+--------------------------------------+---------------------------------+--------------------------------+



In [68]:
# ── 2. AUDIT — know what's broken before touching anything ────────────────────
print("\n── Null count per column ──────────────────────────────")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()
 



── Null count per column ──────────────────────────────
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|       11611894|            0|  11611894|          11611894

In [69]:
total      = df.count()
unique_ids = df.dropDuplicates(["tpep_dropoff_datetime"]).count()
distinct_ids = df.select("tpep_dropoff_datetime").distinct().count()
print(f"total: {total} drop-duplicate {unique_ids} distinct {distinct_ids}")

total: 48722602 drop-duplicate 21616596 distinct 21616596


In [70]:
# ── 3. CLEAN ──────────────────────────────────────────────────────────────────
experiment = df.filter( F.col("tpep_pickup_datetime").between("2025-01-01 00:00:00", "2025-12-31 23:59:59"))
experiment = experiment.filter( F.col("tpep_dropoff_datetime").between("2025-01-01 00:00:00", "2025-12-31 23:59:59"))
                        
experiment = experiment \
    .dropna(subset=["VendorID"]) \
    .fillna({"passenger_count": -1}) \
    .withColumn("total_amount_thb",      F.round(F.col("total_amount") * 36, 2)) \
    .withColumn("is_valid_passenger_count", F.col("passenger_count") > 0) \
    .withColumn("tpep_pickup_date", F.col("tpep_pickup_datetime").cast("date"))
    
 
print(f"\n[CLEANED] Rows: {experiment.count()}")
experiment.printSchema()
experiment.show(10, truncate=False) # truncate=False : show long string

    #.withColumn("email",           F.trim(F.lower(F.col("email")))) \
    #.withColumn("phone",           F.regexp_replace(F.trim(F.col("phone")), r"[^0-9+]", "")) \
    #.withColumn("is_valid_age",    (F.col("age") > 0) & (F.col("age") < 100)) \

#dt.where(dt["product_id"].rlike("^[0-9]{5}$")) # product id must be 5 numbers 
#dt.subtract(dt["product_id"].rlike("^[0-9]{5}$")) # filter out other cases 
#dt.withColumn('product_id', f.substring('product_id', 1,5)) # replace with 5 numbers
#dt.drop("customer_country").withColumnRenamed('customer_country_update', 'customer_country') drop old customer_country and replace it with new renamed customer_country_update


[CLEANED] Rows: 48721798
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = false)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- total_amount_thb: double (nullable = true)
 |-- is_valid_passenger_count: boolean (nulla

In [71]:
# ── 4. VALIDATE ───────────────────────────────────────────────────────────────
print("\n── Rows with invalid salary ───────────────────────────")
experiment.filter(~F.col("is_valid_passenger_count")).select("VendorID", "trip_distance", "total_amount_thb").show()
 
print("\n── Dept distribution after cleaning ───────────────────")
experiment.groupBy("VendorID").count().orderBy("VendorID").show()


── Rows with invalid salary ───────────────────────────
+--------+-------------+----------------+
|VendorID|trip_distance|total_amount_thb|
+--------+-------------+----------------+
|       1|          1.5|           736.2|
|       1|          1.2|           619.2|
|       1|          2.2|           896.4|
|       1|         18.1|         3003.84|
|       1|          7.8|          1765.8|
|       1|          7.3|          1398.6|
|       1|          3.0|           858.6|
|       1|         17.5|          3304.8|
|       1|          1.3|          589.68|
|       1|          1.3|           628.2|
|       1|         17.6|         3003.84|
|       1|         18.0|          3961.8|
|       1|         13.3|          2316.6|
|       1|         13.4|         2366.64|
|       1|          1.7|           585.0|
|       1|         26.7|          4690.8|
|       2|         2.69|           788.4|
|       1|          2.4|           667.8|
|       1|          1.5|           556.2|
|       1|         

In [72]:
# ── 5. WRITE PARTITIONED PARQUET TO MINIO(S3) ─────────────────────────────────────
# Partitioned by dept_id — queries filtering dept skip unrelated folders entirely.
# Spark calls this "partition pruning" — you'll see it in .explain() as PartitionFilters.
output_path = "s3a://spark-warehouse/raw_data/save_from_notebook"


# .parquet(path) = .format("parquet").save(path)
experiment.limit(300000).write \
    .mode("overwrite") \
    .partitionBy("tpep_pickup_date") \
    .parquet(output_path)
 
print(f"\n✓ Written to {output_path}")
print("  Browse in MinIO Console: localhost:9001 → raw-data/output/employees_clean/")
print("  You should see folders: dept_id=ENG/, dept_id=HR/, etc.")
 
# Verify: read back and check row count
verify = spark.read.parquet(output_path)
print(f"\n  Verify read-back: {verify.count()} rows, {len(verify.columns)} cols")


✓ Written to s3a://spark-warehouse/raw_data/save_from_notebook
  Browse in MinIO Console: localhost:9001 → raw-data/output/employees_clean/
  You should see folders: dept_id=ENG/, dept_id=HR/, etc.

  Verify read-back: 300000 rows, 22 cols


**exercise1**

In [73]:
experiment.limit(300000).coalesce(1).write.mode("overwrite") \
         .parquet("s3a://spark-warehouse/raw_data/test_coalesce")

- partition by date created two folder for two days
- coalesce(1) created only a file

**coalesce(1)**    
pros 
- single output file: instead of having too many small files - often for legacy downstream systems or manual human review
- minimized shuffle: avoid shuffle many small partitions - work like repartition(1) but better as it uses narrow dependencies when possible
- reduced metadata overhead: for downstream systems like hdfs, s3 that perform better with large files

cons
- parallelism killer: not leverage distributed cluster
- OOM issue: when reading data again
- upstream pushdown: spark often pushes coalesce(1) as far back in the plan as possible - entire transformation logic might end up running on a single partition, drastically increasing execution time

**exercise2**

In [74]:
partitioned = spark.read.parquet(output_path)
partitioned.filter(F.col("tpep_pickup_date") == "2025-05-02").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#7809,tpep_pickup_datetime#7810,tpep_dropoff_datetime#7811,passenger_count#7812L,trip_distance#7813,RatecodeID#7814L,store_and_fwd_flag#7815,PULocationID#7816,DOLocationID#7817,payment_type#7818L,fare_amount#7819,extra#7820,mta_tax#7821,tip_amount#7822,improvement_surcharge#7823,total_amount#7824,congestion_surcharge#7825,Airport_fee#7826,cbd_congestion_fee#7827,total_amount_thb#7828,is_valid_passenger_count#7829,tpep_pickup_date#7830] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-warehouse/raw_data/save_from_notebook], PartitionFilters: [isnotnull(tpep_pickup_date#7830), (tpep_pickup_date#7830 = 2025-05-02)], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [75]:
partitioned.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#7809,tpep_pickup_datetime#7810,tpep_dropoff_datetime#7811,passenger_count#7812L,trip_distance#7813,RatecodeID#7814L,store_and_fwd_flag#7815,PULocationID#7816,DOLocationID#7817,payment_type#7818L,fare_amount#7819,extra#7820,mta_tax#7821,tip_amount#7822,improvement_surcharge#7823,total_amount#7824,congestion_surcharge#7825,Airport_fee#7826,cbd_congestion_fee#7827,total_amount_thb#7828,is_valid_passenger_count#7829,tpep_pickup_date#7830] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-warehouse/raw_data/save_from_notebook], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




PartitionFilters has value when using with filter

In [76]:
# ── HIVE CHECK ────────────────────────────────────────────────────────────────
# local_db is auto-created by init-hive-db container on first startup
print("\n── Hive databases (local_db should appear) ────────────────────────────")
spark.sql("SHOW DATABASES").show()


── Hive databases (local_db should appear) ────────────────────────────
+---------+
|namespace|
+---------+
|  default|
| local_db|
+---------+



In [77]:
# hive table does not support timestamp_ntz
# we need to cast to timestamp type
experiment = experiment.sort("tpep_pickup_date")
experiment = experiment.withColumn("tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp"))
experiment = experiment.withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))

In [80]:
# ── 6. REGISTER AS HIVE TABLE ─────────────────────────────────────────────────
# Hive stores the table definition (columns, location) in Postgres metastore.
# Any future session can now query it by name — no file path needed.
spark.sql("USE local_db")
spark.sql("DROP TABLE IF EXISTS local_db.sample_hive_table")

# .mode("append")
experiment.limit(300000) \
    .write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("tpep_pickup_date") \
    .saveAsTable("local_db.sample_hive_table")
 
print("\n── Hive table created ─────────────────────────────────")
spark.sql("SHOW TABLES IN local_db").show()



── Hive table created ─────────────────────────────────
+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
| local_db|sample_hive_table|      false|
+---------+-----------------+-----------+



**exercise3**    
with creating temp table, data only persists within a session     
but hive keeps data despite starting a new session

In [81]:
spark.sql("""
    SELECT tpep_pickup_date, COUNT(*) AS headcount
    FROM local_db.sample_hive_table
    GROUP BY tpep_pickup_date
    ORDER BY tpep_pickup_date
""").show()

+----------------+---------+
|tpep_pickup_date|headcount|
+----------------+---------+
|      2025-01-01|    90188|
|      2025-01-02|    84832|
|      2025-01-03|    91250|
|      2025-01-04|    33730|
+----------------+---------+



**exercise4**    
for csv type that doesn't tell spark its schema like parquet, spark will choose data type by itself if no inferSchema=True in read command    
- inferSchema=True will guess string by default, moreover, it performs slowly as it has to scan all data
- explicit schema or StructType performs faster and makes confusing value null


In [82]:
spark.stop()
print("\nDone — local_db.table registered in Hive, data in MinIO")


Done — local_db.table registered in Hive, data in MinIO
